# Off-line Granule Assay NIR Paired Comparison - Simulated

## Purpose

This notebook simulates an off-line granule assay NIR paired comparison for teaching and study-design exploration. Users edit one configuration cell to define acceptance criteria and simulation assumptions before the notebook generates granule metadata, paired NIR values, and reference results.

## Method-specific context

The simulated granule workflow represents a paired study where the same granule sample presentation is measured by old and changed off-line NIR methods. Simulated metadata include batch, granule lot, vial, refill replicate, analyst, instruments, and date so report tables resemble the fixed-data mother notebook.

In [1]:
# USER CONFIGURATION - edit values below
METHOD_DISPLAY_NAME = "Simulated Off-line Granule Assay NIR"
D_EQUIVALENCE_MARGIN = 1.0
K_PRECISION_RATIO = 2.0
ALPHA_ACCURACY = 0.05
ALPHA_PRECISION = 0.05
RECOMMENDED_MIN_N = 18

N_SAMPLES = 24
RANDOM_SEED = 12345
N_SIM = 500

OLD_MEAN = 100.0
OLD_SD = 0.9
NEW_MEAN = 101
NEW_SD = 0.85

REFERENCE_SD = 0.25
SAMPLE_TO_SAMPLE_SD = 0.8

PRECISION_METHOD = "observed_sd_ratio_exploratory"
OLD_VARIANCE = None
ALLOW_EXPLORATORY_PRECISION_AS_PRIMARY = True
EXPORT_REPORT = False


In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from nir_cp.notebook_export import export_notebook_pdf
from nir_cp.paired_comparison import paired_comparison_decision
from nir_cp.plots import plot_bland_altman, plot_difference_vs_reference, plot_old_vs_new
from nir_cp.report_equations import display_equation, display_equation_set
from nir_cp.report_tables import display_report_dataframe
from nir_cp.reporting_text import paired_decision_summary_text
from nir_cp.simulation import (
    simulate_granule_assay_study_dataframe,
    simulate_paired_comparison_success,
    simulate_paired_study_dataframe,
    simulate_tablet_transmission_study_dataframe,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

METHOD_NAME = METHOD_DISPLAY_NAME


## Simulate data

In [3]:
df = simulate_granule_assay_study_dataframe(
    n_samples=N_SAMPLES,
    old_mean=OLD_MEAN,
    old_sd=OLD_SD,
    new_mean=NEW_MEAN,
    new_sd=NEW_SD,
    sample_to_sample_sd=SAMPLE_TO_SAMPLE_SD,
    reference_sd=REFERENCE_SD,
    seed=RANDOM_SEED,
)

display_report_dataframe(df, max_rows=10);
display_report_dataframe(df[["batch_id", "granule_lot", "refill_replicate", "vial_id"]], max_rows=10);

sample_id,batch_id,granule_lot,vial_id,old_nir,new_nir,reference,refill_replicate,analyst,instrument_old,instrument_new,date
G-SYN-001,G-B001,G-L001,V-001,98.719,99.913,98.863,1,GA01,NIR-OLD-GRANULE,NIR-NEW-GRANULE,2026-03-01
G-SYN-002,G-B001,G-L001,V-002,101.416,102.071,101.145,2,GA02,NIR-OLD-GRANULE,NIR-NEW-GRANULE,2026-03-02
G-SYN-003,G-B001,G-L001,V-003,98.094,100.672,99.037,3,GA03,NIR-OLD-GRANULE,NIR-NEW-GRANULE,2026-03-03
G-SYN-004,G-B001,G-L001,V-004,99.719,101.029,99.747,1,GA01,NIR-OLD-GRANULE,NIR-NEW-GRANULE,2026-03-04
G-SYN-005,G-B001,G-L001,V-005,101.492,101.390,100.345,2,GA02,NIR-OLD-GRANULE,NIR-NEW-GRANULE,2026-03-05
G-SYN-006,G-B001,G-L001,V-006,101.764,100.864,99.328,3,GA03,NIR-OLD-GRANULE,NIR-NEW-GRANULE,2026-03-06
G-SYN-007,G-B001,G-L002,V-007,99.605,100.431,98.702,1,GA01,NIR-OLD-GRANULE,NIR-NEW-GRANULE,2026-03-07
G-SYN-008,G-B001,G-L002,V-008,101.265,100.843,100.616,2,GA02,NIR-OLD-GRANULE,NIR-NEW-GRANULE,2026-03-08
G-SYN-009,G-B002,G-L002,V-009,99.426,101.544,100.233,3,GA03,NIR-OLD-GRANULE,NIR-NEW-GRANULE,2026-03-09
G-SYN-010,G-B002,G-L002,V-010,97.349,98.075,98.262,1,GA01,NIR-OLD-GRANULE,NIR-NEW-GRANULE,2026-03-10


batch_id,granule_lot,refill_replicate,vial_id
G-B001,G-L001,1,V-001
G-B001,G-L001,2,V-002
G-B001,G-L001,3,V-003
G-B001,G-L001,1,V-004
G-B001,G-L001,2,V-005
G-B001,G-L001,3,V-006
G-B001,G-L002,1,V-007
G-B001,G-L002,2,V-008
G-B002,G-L002,3,V-009
G-B002,G-L002,1,V-010


## Data checks

In [4]:
required_columns = ['sample_id', 'batch_id', 'granule_lot', 'vial_id', 'old_nir', 'new_nir', 'reference', 'refill_replicate', 'analyst', 'instrument_old', 'instrument_new', 'date']
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
if df[required_columns].isna().any().any():
    raise ValueError("Simulated data contain missing values in required columns.")
if df["sample_id"].duplicated().any():
    raise ValueError("Sample identifiers must be unique for this paired workflow.")
for column in ["old_nir", "new_nir", "reference"]:
    df[column] = pd.to_numeric(df[column], errors="raise")

df["difference_new_minus_old"] = df["new_nir"] - df["old_nir"]
df["average_old_new"] = (df["old_nir"] + df["new_nir"]) / 2

display_report_dataframe(
    pd.DataFrame([
        {
            "n_pairs": len(df),
            "recommended_min_n": RECOMMENDED_MIN_N,
            "mean_old_nir": df["old_nir"].mean(),
            "mean_new_nir": df["new_nir"].mean(),
            "mean_difference_new_minus_old": df["difference_new_minus_old"].mean(),
            "sd_old_nir": df["old_nir"].std(ddof=1),
            "sd_new_nir": df["new_nir"].std(ddof=1),
        }
    ]),
    transpose_if_one_row=True,
);

parameter,value
n_pairs,24.000
recommended_min_n,18.000
mean_old_nir,100.107
mean_new_nir,100.792
mean_difference_new_minus_old,0.685
sd_old_nir,1.229
sd_new_nir,1.176


# Statistical theory and decision rules

## Study design

The same physical sample is measured with the current/old and changed/new NIR procedure. The paired difference is defined as:

In [5]:
_ = display_equation(r"D_i = Y_{N,i} - Y_{O,i}")

Pairing removes between-sample variation from the accuracy comparison. This notebook calls tested functions under `src/nir_cp/` and does not implement independent pass/fail logic.

## Accuracy equivalence

The paired accuracy hypotheses are:

In [6]:
_ = display_equation_set([
    r"H_0: |\mu_D| \ge d",
    r"H_A: |\mu_D| < d",
])

The sample mean and variance of the paired differences are:

In [7]:
_ = display_equation_set([
    r"\bar{D} = \frac{1}{n}\sum_{i=1}^{n}D_i",
    r"s_D^2 = \frac{\sum_{i=1}^{n}(D_i-\bar{D})^2}{n-1}",
])

The confidence interval used for paired accuracy is:

In [8]:
_ = display_equation(r"\bar{D} \pm t_{1-\alpha,n-1}\frac{s_D}{\sqrt{n}}")

For `alpha = 0.05`, this is a two-sided 90% confidence interval used for TOST equivalence. The accuracy criterion is met if the full confidence interval is contained within:

In [9]:
_ = display_equation(r"[-d, +d]")

## Precision noninferiority

The precision hypotheses are:

In [10]:
_ = display_equation_set([
    r"H_0: \frac{\sigma_N}{\sigma_O} \ge k",
    r"H_A: \frac{\sigma_N}{\sigma_O} < k",
])

Precision assessment depends on the study design and available variance information. For heterogeneous paired samples, the observed raw SD ratio is supportive/exploratory unless explicitly justified in the CP. For the known old-variance option, the upper bound is:

In [11]:
_ = display_equation(r"U = \sqrt{\frac{(n-1)s_D^2}{\sigma_O^2\chi^2_{\alpha,n-1}} - 1}")

with decision rule:

In [12]:
_ = display_equation(r"U < k")

For the duplicate-measurement option, the duplicate scaled differences are:

In [13]:
_ = display_equation_set([
    r"d_{O,i} = \frac{Y_{O,i,1}-Y_{O,i,2}}{\sqrt{2}}",
    r"d_{N,i} = \frac{Y_{N,i,1}-Y_{N,i,2}}{\sqrt{2}}",
])

and an F-based one-sided upper confidence bound is used for `sigma_N / sigma_O`.

> Precision method note: the observed SD-ratio calculation shown in this example is supportive/exploratory for heterogeneous paired samples unless specifically justified in the CP. Final CP/GMP use should apply the CP-approved primary precision method.

## Values used in this notebook

In [14]:
values_used = {
    "method name": METHOD_NAME,
    "precision_method": PRECISION_METHOD,
    "n": len(df),
    "recommended_min_n": RECOMMENDED_MIN_N,
    "d": D_EQUIVALENCE_MARGIN,
    "k": K_PRECISION_RATIO,
    "alpha_accuracy": ALPHA_ACCURACY,
    "alpha_precision": ALPHA_PRECISION,
    "old_mean_assumption": OLD_MEAN,
    "old_sd_assumption": OLD_SD,
    "new_mean_assumption": NEW_MEAN,
    "new_sd_assumption": NEW_SD,
    "sample_to_sample_sd": SAMPLE_TO_SAMPLE_SD,
    "reference_sd": REFERENCE_SD,
    "random seed": RANDOM_SEED,
    "n_sim": N_SIM,
}
if OLD_VARIANCE is not None:
    values_used["old_variance"] = OLD_VARIANCE

display_report_dataframe(pd.DataFrame([values_used]), transpose_if_one_row=True);

parameter,value
method name,Simulated Off-line Granule Assay NIR
precision_method,observed_sd_ratio_exploratory
n,24
recommended_min_n,18
d,1.000
k,2.000
alpha_accuracy,0.050
alpha_precision,0.050
old_mean_assumption,100.000
old_sd_assumption,0.900


## Paired comparison decision

In [15]:
decision = paired_comparison_decision(
    old_values=df["old_nir"],
    new_values=df["new_nir"],
    d=D_EQUIVALENCE_MARGIN,
    k=K_PRECISION_RATIO,
    alpha_accuracy=ALPHA_ACCURACY,
    alpha_precision=ALPHA_PRECISION,
    precision_method=PRECISION_METHOD,
    old_variance=OLD_VARIANCE,
    allow_exploratory_precision_as_primary=ALLOW_EXPLORATORY_PRECISION_AS_PRIMARY,
)

display(Markdown(paired_decision_summary_text(decision, method_name=METHOD_NAME)))
display_report_dataframe(pd.DataFrame([decision["accuracy"]]), transpose_if_one_row=True);
display_report_dataframe(pd.DataFrame([decision["precision"]]), transpose_if_one_row=True);
display_report_dataframe(pd.DataFrame([{"overall_pass": decision["overall_pass"], "decision_text": decision["decision_text"]}]), transpose_if_one_row=True);

Simulated Off-line Granule Assay NIR did not meet the predefined criteria for paired accuracy equivalence and precision noninferiority. The paired mean difference (new - old) was 0.685, with a 90.0% confidence interval from 0.317 to 1.053; the predefined equivalence margin was +/-1.000. The upper confidence bound for the precision ratio (sigma_new / sigma_old) was 1.358, compared with the predefined limit k=2.000.

parameter,value
pass,False
d,1.000
alpha,0.050
ci_confidence,0.900
mean_difference,0.685
lower,0.317
upper,1.053
n,24


parameter,value
sd_old,1.229
sd_new,1.176
ratio_observed,0.957
upper_bound,1.358
alpha,0.050
n_old,24
n_new,24
df_old,23
df_new,23
method,observed_sd_ratio_exploratory


parameter,value
overall_pass,False
decision_text,The changed NIR method did not meet the predefined paired accuracy equivalence and exploratory observed-SD precision criteria. This precision method is supportive/exploratory and is not the preferred USP <1010> paired precision method for heterogeneous samples unless justified.


## Supporting plots

In [16]:
display(plot_old_vs_new(df, sample_id_col="sample_id"))
display(plot_bland_altman(df))
display(plot_difference_vs_reference(df))

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

## Probability-of-success simulation

In [17]:
simulation_result = simulate_paired_comparison_success(
    n=len(df),
    true_bias=NEW_MEAN - OLD_MEAN,
    old_sd=OLD_SD,
    new_sd=NEW_SD,
    d=D_EQUIVALENCE_MARGIN,
    k=K_PRECISION_RATIO,
    n_sim=N_SIM,
    seed=RANDOM_SEED,
    alpha_accuracy=ALPHA_ACCURACY,
    alpha_precision=ALPHA_PRECISION,
)
display_report_dataframe(pd.DataFrame([simulation_result]), transpose_if_one_row=True);

parameter,value
n,24.000
true_bias,1.000
old_sd,0.900
new_sd,0.850
d,1.000
k,2.000
n_sim,500.000
seed,12345.000
alpha_accuracy,0.050
alpha_precision,0.050


## Export report

In [18]:
if EXPORT_REPORT:
    notebook_path = PROJECT_ROOT / "notebooks" / "02_granule_assay_paired_comparison_simulated.ipynb"
    pdf_path = PROJECT_ROOT / "reports" / "pdf" / "02_granule_assay_paired_comparison_simulated.pdf"
    exported_pdf = export_notebook_pdf(notebook_path, pdf_path, hide_code=True, keep_html=True)
    display(Markdown(f"Exported PDF report: `{exported_pdf}`"))
else:
    display(Markdown("PDF export skipped because `EXPORT_REPORT` is `False`."))

PDF export skipped because `EXPORT_REPORT` is `False`.